# Colorectal Cancer (CRC) - Exploratory Data Analysis (EDA)
## Discovery Cohort: GSE39582 (Affymetrix HG-U133 Plus 2.0)

**Objective:**
1. Inspect clinical demographics, TNM tumor staging, and adjuvant chemotherapy.
2. Characterize disease recurrence events and relapse-free survival (RFS) time distributions.
3. Formulate the primary classification cohort: Early Recurrence (<= 1 yr) vs Long-Term Disease-Free (>= 5 yr).
4. Verify expression matrix normalization (RMA log2 scale) and sample distributions.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Publication style styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 120

# Paths
DATA_DIR = Path('../data')
CLINICAL_PATH = DATA_DIR / 'processed' / 'clinical' / 'GSE39582_clinical_raw.csv'
EXPR_PARQUET = DATA_DIR / 'processed' / 'expression' / 'GSE39582_expression.parquet'
FIGURES_DIR = Path('../results') / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Clinical Metadata

In [ ]:
clin_df = pd.read_csv(CLINICAL_PATH, index_col='gsm_id')
print(f"Total samples in clinical metadata: {len(clin_df)}")
clin_df.head()

## 2. Inspect Clinical Variables & Missingness

In [ ]:
missing_df = pd.DataFrame({
    'Total': len(clin_df),
    'Non-Null': clin_df.notnull().sum(),
    'Missing': clin_df.isnull().sum(),
    '% Missing': (clin_df.isnull().sum() / len(clin_df) * 100).round(1)
})
missing_df.sort_values(by='Missing', ascending=False)

## 3. Tumor Stage & Demographic Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Stage distribution
if 'stage' in clin_df.columns or 'tumor_stage' in clin_df.columns:
    stage_col = 'stage' if 'stage' in clin_df.columns else 'tumor_stage'
    clin_df[stage_col].value_counts().plot(kind='bar', ax=axes[0], color='teal', edgecolor='black')
    axes[0].set_title('Colorectal Cancer Stage Distribution')
    axes[0].set_ylabel('Sample Count')
    axes[0].set_xlabel('Stage')

# Sex distribution
if 'sex' in clin_df.columns or 'gender' in clin_df.columns:
    sex_col = 'sex' if 'sex' in clin_df.columns else 'gender'
    clin_df[sex_col].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'])
    axes[1].set_title('Patient Sex Distribution')
    axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_clinical_demographics.png', dpi=300)
plt.show()

## 4. Recurrence and Survival Endpoint Analysis

In [ ]:
# Inspect relapse columns
recurrence_cols = [c for c in clin_df.columns if any(k in c.lower() for k in ['relapse', 'rfs', 'dfs', 'recur'])]
print("Identified recurrence/survival columns:", recurrence_cols)
clin_df[recurrence_cols].describe(include='all')

## 5. Expression Matrix Dynamic Range Inspection

In [ ]:
if EXPR_PARQUET.exists():
    expr_df = pd.read_parquet(EXPR_PARQUET)
    print(f"Expression Matrix Loaded: {expr_df.shape[0]:,} probes x {expr_df.shape[1]:,} samples")
    
    # Sample expression values
    subset_vals = expr_df.iloc[:2000, :10].values.flatten()
    plt.figure(figsize=(8, 4))
    sns.histplot(subset_vals, kde=True, bins=50, color='darkblue')
    plt.title('Gene Expression Value Distribution (RMA Log2 Verification)')
    plt.xlabel('Log2 Expression Intensity')
    plt.ylabel('Density / Frequency')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_expression_distribution.png', dpi=300)
    plt.show()
else:
    print("Parquet expression matrix not yet created. Run src/data_loader.py first.")